In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS files.amazon_files")

DataFrame[]

In [0]:
# Criando e lendo o dataset da Amazon com o Spark SQL

df = spark.read.csv(
    "/Volumes/files/amazon_files/volumes/amazon.csv",
    header=True,
    inferSchema=True
)

df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: string (nullable = true)
 |-- actual_price: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- about_product: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_content: string (nullable = true)
 |-- img_link: string (nullable = true)
 |-- product_link: string (nullable = true)



In [0]:
# Agrupando os dados por rating e contando a quantidade de vezes que cada rating aparece

df.groupBy("rating").count().orderBy("rating", ascending=False).show()

+--------------------+-----+
|              rating|count|
+--------------------+-----+
|                 ₹90|    1|
|              ₹9,999|    1|
|              ₹9,990|    1|
|              ₹7,999|    1|
|              ₹3,999|    3|
|              ₹2,499|    1|
|              ₹1,999|    2|
|              ₹1,998|    2|
|              ₹1,799|    5|
|              ₹1,599|    2|
|              ₹1,500|    1|
|                   ||    1|
|      in-Built Games|    1|
|In-Built Games(Ch...|    1|
|Electronics|Weara...|    8|
|Electronics|Mobil...|    1|
|                 63%|    1|
|                 60%|    1|
|                 5.0|    3|
|                 4.8|    3|
+--------------------+-----+
only showing top 20 rows


In [0]:
# Verificando se existem ratings fora do padrão
# rlike = regex para filtrar strings
# O que é regex = expressões regulares https://www.regular-expressions.info/
# F.col = colunas do dataframe https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.sql.
# ~F.col = not = não, ao inverso do que está na função
# filter = filtrar
# r"^[0-9]\.[0-9]$ = regex para filtrar strings que começam com um número de 0 a 9, seguido de um ponto e um número de 0 a 9
# count() = contar

from pyspark.sql import functions as F

df.filter(~F.col("rating").rlike(r"^[0-9]\.[0-9]$")).count()

112

In [0]:
df.select("review_content").show()

+--------------------+
|      review_content|
+--------------------+
|Looks durable Cha...|
|I ordered this ca...|
|Not quite durable...|
|Good product,long...|
|Bought this inste...|
|It's a good produ...|
|Build quality is ...|
|Worth for money -...|
|I use this to con...|
|I ordered this ca...|
|The cable is effi...|
|Good product,long...|
|I am using it for...|
|Definitely isn’t ...|
|Loosing charging ...|
|I like it 👍👍,Be...|
|Pros- xiomi 5a is...|
|I ordered this ca...|
|Check for offera ...|
|LG was always Goo...|
+--------------------+
only showing top 20 rows


In [0]:
# .select = selecionar colunas
# .show = mostrar
# .count = contar
# .orderBy = ordenar
# .desc = decrescente
# .asc = crescente

from pyspark.sql import functions as F

df.filter(~F.col("rating").rlike(r"^[0-9]\.[0-9]$")) \
    .select("product_id", "category", "rating", "about_product") \
        .show(5, truncate=False)

+----------+--------------------+-------------------------------------+-------------------------------------------+
|product_id|category            |rating                               |about_product                              |
+----------+--------------------+-------------------------------------+-------------------------------------------+
|B088Z1YWBC| AV                 | Audio Out | (E03i31 / E04i32) Black"|₹9,490                                     |
|B0BLV1GNLN| PC                 |₹9,990                               |4.0                                        |
|B0BF57RN3K| AI Voice Assistance|₹1,799                               |91%                                        |
|B0B3RRWSF6|120+ Sports Modes   |₹1,998                               |80%                                        |
|B0B5LVS732| 550 NITS           | 100 Sports Mode with Auto Detection |Electronics|WearableTechnology|SmartWatches|
+----------+--------------------+-------------------------------------+-

In [0]:
df_filtrado = df.filter(F.col("rating").rlike(r"^[0-9]\.[0-9]$"))
df_filtrado.count()

1353

In [0]:
df_filtrado_agrupado = df.filter(
    (~F.col("discounted_price").rlike(r"^₹"))
    | (~F.col("actual_price").rlike(r"^₹"))
    | (~F.col("discount_percentage").rlike(r"%$"))
)


df_filtrado_agrupado.count()

57

In [0]:
df_todas_ruins = df.filter(
    (~F.col("rating").rlike(r"^[0-9]\.[0-9]$")) |
    (~F.col("discounted_price").rlike(r"^₹")) |
    (~F.col("actual_price").rlike(r"^₹")) |
    (~F.col("discount_percentage").rlike(r"%$"))

)

df_todas_ruins.count()

112